# 📥 Notebook 02 — MySQL Bulk Insertion & Performance Benchmarking

**Project:** Million Records MySQL Performance Benchmarking System  
**Author:** Backend Engineering Team  
**Purpose:** Insert employee records into MySQL using `executemany()` with varying batch sizes, measuring insertion time, throughput, CPU, and memory usage.

---

## 🎯 What This Notebook Does
1. Connects to MySQL and creates the `performance_test.employees` table
2. Reads `data/employees.csv` into a DataFrame
3. Runs bulk insertion benchmarks across **4 dataset sizes × 4 batch sizes**
4. Captures: insertion time, throughput (rec/s), CPU %, memory MB
5. Saves all results to `data/results.csv` for analysis in Notebook 03

## ⚙️ Benchmark Matrix
| Dataset Sizes | Batch Sizes          |
|---------------|----------------------|
| 10K, 100K, 500K, 1M | 100, 1000, 5000, 10000 |

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Imports & Configuration
# ─────────────────────────────────────────────────────────────────────────────

import time
import csv
import psutil
import pandas as pd
import mysql.connector
from mysql.connector import Error
from tqdm import tqdm

# ── File Paths ──────────────────────────────────────────────────────────────
EMPLOYEES_CSV = "employees.csv"
RESULTS_CSV   = "results.csv"

# ── MySQL Config ────────────────────────────────────────────────────────────
DB_CONFIG = {
    'host': 'localhost',
    'port': 3306,
    'user': 'root',
    'password': 'Utkarshahu@18',   # apna password
    'database': 'performance_test'
}

# ── Benchmark Parameters ────────────────────────────────────────────────────
DATASET_SIZES = [10000, 100000]
BATCH_SIZES = [100, 1000, 5000]

print("✅ Configuration Loaded")
print(f"📁 Employees CSV: {EMPLOYEES_CSV}")
print(f"📁 Results CSV: {RESULTS_CSV}")

✅ Configuration Loaded
📁 Employees CSV: employees.csv
📁 Results CSV: results.csv


In [4]:
# Connect MySQL

try:

    conn = mysql.connector.connect(**DB_CONFIG)

    if conn.is_connected():

        print("✅ Connected To MySQL")

except Error as e:

    print("❌ Error:", e)

✅ Connected To MySQL


In [5]:
# Read CSV

df = pd.read_csv(EMPLOYEES_CSV)

print(df.head())

print(f" Total Records Loaded: {len(df)}")

              name                      email          city  salary
0     Michael Wong   travisholden@example.com    South Cody   46190
1    Gregory Welch         ascott@example.org  Lake Bradley   97955
2  Janet Rodriguez    ericmahoney@example.net    Conleystad  195400
3   Kelly Mitchell    colecabrera@example.net     Johnburgh  196972
4     Alyssa Myers  jeremyvasquez@example.com  West Timothy   70820
 Total Records Loaded: 1798205


In [8]:
import pandas as pd
import mysql.connector
import time

# Read CSV
df = pd.read_csv("employees.csv")

# Remove NaN rows
df = df.dropna()


# Convert salary to int
df['salary'] = pd.to_numeric(
    df['salary'],
    errors='coerce'
)

# Remove invalid salary rows
df = df.dropna(subset=['salary'])

# Convert salary to int
df['salary'] = df['salary'].astype(int)

# MySQL Connection
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Utkarshahu@18",
    database="performance_test"
)

cursor = conn.cursor()

# Insert Query
query = """
INSERT INTO employees(name, email, city, salary)
VALUES (%s, %s, %s, %s)
"""

# Convert rows
values = list(
    df.itertuples(index=False, name=None)
)

# Start timer
start_time = time.time()

# Bulk insert
cursor.executemany(query, values)

# Commit
conn.commit()

# End timer
end_time = time.time()

# Results
print("✅ Data Inserted Successfully")

print(f"📊 Total Rows: {len(values)}")

print(f"⏱ Time Taken: {end_time - start_time:.2f} sec")

# Close
cursor.close()
conn.close()

OperationalError: 2013 (HY000): Lost connection to MySQL server during query

In [ ]:
throughput = len(df) / total_time

print(f"Throughput: {throughput:.2f} records/sec")

Throughput: 52632.70 records/sec


In [ ]:
cpu = psutil.cpu_percent()

memory = psutil.virtual_memory().percent

print(f"🖥 CPU Usage: {cpu}%")
print(f"💾 Memory Usage: {memory}%")

🖥 CPU Usage: 25.6%
💾 Memory Usage: 87.9%


In [1]:
results = pd.DataFrame([{
    "records": len(df),
    "time": total_time,
    "throughput": throughput,
    "cpu": cpu,
    "memory": memory
}])

results.to_csv(RESULTS_CSV, index=False)

print("✅ Results Saved")

NameError: name 'pd' is not defined